In [1]:
import os

# Fix protobuf incompatibility with TensorFlow
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization
from tensorflow.keras import optimizers
from tensorflow.keras import backend as K

TRAIN_PATH = os.getenv(
    "TRAIN_PATH", os.path.join("data", "train.csv")
)  # fallback to relative path
TEST_PATH = os.getenv(
    "TEST_PATH", os.path.join("data", "test.csv")
)  # fallback to relative path

# Use a larger sample for better learning while staying memory‑friendly
DATASET_SIZE = int(os.getenv("DATASET_SIZE", "500000"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "0.001"))
EPOCHS = int(os.getenv("EPOCHS", "20"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "1024"))
SUBMISSION_NAME = os.getenv("SUBMISSION_NAME", "submission.csv")

train_full = pd.read_csv(TRAIN_PATH, low_memory=False)
test_full = pd.read_csv(TEST_PATH, low_memory=False)

if len(train_full) > DATASET_SIZE:
    train_full = train_full.sample(n=DATASET_SIZE, random_state=42).reset_index(
        drop=True
    )

train_df, val_df = train_test_split(train_full, test_size=0.5, random_state=1)
val_df = val_df[:10000]  # keep validation small for quick evaluation

testKaggle = test_full.copy()




2026-05-04 19:59:38.106932: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777924778.120403      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777924778.124510      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-04 19:59:38.140704: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

In [2]:
def clean(df):
    """Basic cleaning: drop NaNs, remove obvious outliers."""
    df = df.dropna()
    # Only filter fare_amount if the column exists (test set lacks it)
    if "fare_amount" in df.columns:
        df = df[df["fare_amount"] > 0]
    lon_min, lon_max = -80, -70
    lat_min, lat_max = 40, 45
    for col in ["pickup_longitude", "dropoff_longitude"]:
        df = df[(df[col] >= lon_min) & (df[col] <= lon_max)]
    for col in ["pickup_latitude", "dropoff_latitude"]:
        df = df[(df[col] >= lat_min) & (df[col] <= lat_max)]
    return df.reset_index(drop=True)


def add_time_features(df):
    """Extract hour, weekday, month from pickup_datetime."""
    dt = pd.to_datetime(df["pickup_datetime"])
    df["hour"] = dt.dt.hour
    df["weekday"] = dt.dt.weekday
    df["month"] = dt.dt.month
    return df


def haversine_distance(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance in kilometres."""
    R = 6371.0
    lat1, lat2 = np.radians(lat1), np.radians(lat2)
    lon1, lon2 = np.radians(lon1), np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


def add_distances_features(df):
    """Add haversine distance and a simple Manhattan approximation."""
    df["haversine"] = haversine_distance(
        df["pickup_latitude"],
        df["pickup_longitude"],
        df["dropoff_latitude"],
        df["dropoff_longitude"],
    )
    df["manhattan"] = np.abs(df["pickup_latitude"] - df["dropoff_latitude"]) + np.abs(
        df["pickup_longitude"] - df["dropoff_longitude"]
    )
    return df


def add_coordinate_features(df):
    """Difference in lat/lon as separate features."""
    df["delta_lat"] = df["dropoff_latitude"] - df["pickup_latitude"]
    df["delta_lon"] = df["dropoff_longitude"] - df["pickup_longitude"]
    return df


print("Cleaning training split")
train_df = clean(train_df)
print("Cleaning validation split")
val_df = clean(val_df)
print("Cleaning test split")
testKaggle = clean(testKaggle)

train_df = add_time_features(train_df)
val_df = add_time_features(val_df)
testKaggle = add_time_features(testKaggle)

train_df = add_coordinate_features(train_df)
val_df = add_coordinate_features(val_df)
testKaggle = add_coordinate_features(testKaggle)

train_df = add_distances_features(train_df)
val_df = add_distances_features(val_df)
testKaggle = add_distances_features(testKaggle)




Cleaning training split


NameError: name 'train_df' is not defined

In [3]:
feature_cols = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
    "hour",
    "weekday",
    "month",
    "delta_lat",
    "delta_lon",
    "haversine",
    "manhattan",
]

train_features = train_df[feature_cols].values
val_features = val_df[feature_cols].values
test_features = testKaggle[feature_cols].values

val_labels_log = np.log1p(val_df["fare_amount"].values)
train_labels_log = np.log1p(train_df["fare_amount"].values)

scaler = preprocessing.MinMaxScaler()
train_scaled = scaler.fit_transform(train_features)
validation_scaled = scaler.transform(val_features)
test_scaled = scaler.transform(test_features)




NameError: name 'train_df' is not defined

In [4]:
def rmse_tf(y_true, y_pred):
    """TensorFlow RMSE metric."""
    return K.sqrt(K.mean(K.square(y_pred - y_true)))


model = Sequential()
model.add(
    Dense(
        256,
        activation="relu",
        input_dim=train_scaled.shape[1],
    )
)
model.add(BatchNormalization())
model.add(Dense(128, activation="relu"))
model.add(BatchNormalization())
model.add(Dense(64, activation="relu"))
model.add(BatchNormalization())
model.add(Dense(32, activation="relu"))
model.add(BatchNormalization())
model.add(Dense(8, activation="relu"))
model.add(BatchNormalization())
model.add(Dense(1, activation="linear"))  # predicts log‑fare

adam = optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(
    loss="mse",
    optimizer=adam,
    metrics=["mae", rmse_tf, "mse"],
)

print("Model summary:")
model.summary()




NameError: name 'train_scaled' is not defined

In [5]:
history = model.fit(
    train_scaled,
    train_labels_log,
    validation_data=(validation_scaled, val_labels_log),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)

val_pred_log = model.predict(validation_scaled).flatten()
val_pred = np.expm1(val_pred_log)  # revert log1p
val_true = np.expm1(val_labels_log)
val_rmse = np.sqrt(((val_pred - val_true) ** 2).mean())
print(f"Validation RMSE (original scale): {val_rmse:.4f}")




NameError: name 'train_scaled' is not defined

In [6]:
predictionKaggle_log = model.predict(test_scaled, batch_size=128, verbose=1).flatten()
predictionKaggle = np.expm1(predictionKaggle_log)

# Ensure predictions are finite and within a realistic range
predictionKaggle = np.where(np.isfinite(predictionKaggle), predictionKaggle, 0.0)
predictionKaggle = np.clip(predictionKaggle, 0.0, 200.0)

keys = testKaggle["key"].values
submission_df = pd.DataFrame({"key": keys, "fare_amount": predictionKaggle})
submission_df.to_csv(SUBMISSION_NAME, index=False)
print(f"Submission file written to {SUBMISSION_NAME}")

NameError: name 'test_scaled' is not defined